# Install Necessary Libraries

In [ ]:
%%capture

!pip install torch transformers chromadb datasets qwen-vl-utils hf_xet langchain-chroma langchain-core Pillow

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("HF_TOKEN is not set. Please add it to Colab secrets.")

Successfully logged in to Hugging Face!


# (Optional) Install Flash Attention

Makes models run faster with less VRAM usage.

**Requires Ampere or newer GPU — T4 does NOT support Flash Attention.**
Skip this cell if running on T4. Use L4 or better for Flash Attention.

In [ ]:
!nvidia-smi

# Uncomment below ONLY if running on Ampere+ GPU (L4, A100, etc.) — NOT T4
# !pip install ninja
# !pip install flash-attn --no-build-isolation

Thu Mar 26 02:09:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Setup Paths to VDB and Qwen Scripts (the new VDB is in the shared folder)

In [ ]:
import os
import sys

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# download this folder from my drive to get the scripts and the preprocessed VDB

# https://drive.google.com/drive/folders/1YOmGkaUuylhh3IjivjrnmoOyQxQ9KX3T?usp=sharing
# VDB is currently embedded with Qwen3VL-Embedding-2B

# when you mount your drive, adjust these paths accordingly to your drive
sys.path.append('/content/drive/MyDrive/colab/mcam/')
VDB_PATH = '/content/drive/MyDrive/colab/mcam/VDB'

Mounted at /content/drive


In [ ]:
import os

repo_path = "/content/Mills-Museum"

if not os.path.exists(repo_path):
    os.system("git clone https://github.com/cs2535-oakhoury-2026spring/Mills-Museum.git /content/Mills-Museum")
else:
    os.system("cd /content/Mills-Museum && git pull")

print("Repo ready.")

Repo ready.


# Choose How Many Keywords to Find

In [ ]:
TERM_COUNT = 10

# Load Data & Vector Store

Loads the LangChain Chroma vectorstore and AAT term-to-ID mapping. No GPU needed for this cell.

In [ ]:
COLLECTION_NAME = "aat_terms"

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from datasets import load_dataset
from datasets.arrow_dataset import Dataset
import torch
import gc
import csv

# Import Qwen wrappers
from scripts.qwen3_vl_embedding import Qwen3VLEmbedder
from scripts.qwen3_vl_reranker import Qwen3VLReranker

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load LangChain Chroma vector store (no embedding_function needed for search-by-vector)
vectorstore = Chroma(
    collection_name="aat_terms",
    persist_directory=VDB_PATH,
    collection_metadata={"hnsw:space": "cosine"},
)
print(f"Loaded vectorstore with {vectorstore._collection.count()} documents")

# Load AAT term-to-ID mapping
aat_terms: Dataset = load_dataset(path="KeeganC/aat-museum-subset", split="train")
selection = aat_terms.select_columns(['preferred_term', 'subject_id'])
aat_terms_to_ids: dict[str, str] = {}
for i in selection.iter(1):
    aat_terms_to_ids[i['preferred_term'][0]] = i['subject_id'][0]

# MMR parameters
FETCH_K = TERM_COUNT * 4  # fetch more candidates for MMR diversity selection
LAMBDA_MULT = 0.7         # 1.0 = pure relevance, 0.0 = pure diversity

print("Data loaded. Ready to run pipeline.")

Using device: cuda
Loaded vectorstore with 44225 documents


README.md:   0%|          | 0.00/739 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/11.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44225 [00:00<?, ? examples/s]

Data loaded. Ready to run pipeline.


# Generate Keywords

Runs embedding + LangChain Chroma MMR retrieval, then reranking.

This cell uses `max_marginal_relevance_search_by_vector` from `langchain_chroma` to
apply MMR diversity selection over the Chroma vectorstore, then reranks with the
Qwen VL Reranker.

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
import gc
import csv

from scripts.qwen3_vl_embedding import Qwen3VLEmbedder
from scripts.qwen3_vl_reranker import Qwen3VLReranker

# Load embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Qwen embedding model on {device}...")

try:
    import flash_attn
    print("Flash Attention found, proceeding with optimized code path.")
    ATTENTION_IMPLEMENTATION = "flash_attention_2"
except ImportError:
    print("Flash Attention not installed, falling back to default attention implementation.")
    ATTENTION_IMPLEMENTATION = "sdpa"

embedding_model_name = "Qwen/Qwen3-VL-Embedding-2B"

embedding_model = Qwen3VLEmbedder(
    model_name_or_path=embedding_model_name,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    attn_implementation=ATTENTION_IMPLEMENTATION
)

# Load reranking model
print(f"Loading Qwen reranking model on {device}...")
reranking_model_name = "Qwen/Qwen3-VL-Reranker-2B"

reranking_model = Qwen3VLReranker(
    model_name_or_path=reranking_model_name,
    dtype=torch.bfloat16,
    attn_implementation=ATTENTION_IMPLEMENTATION
)

print("Both models loaded and ready!")

Loading Qwen embedding model on cuda...
Flash Attention not installed, falling back to default attention implementation.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

Both models loaded and ready!


# API for Frontend


In [ ]:
!pip install fastapi uvicorn pyngrok nest-asyncio python-multipart langchain-chroma langchain-core

import uvicorn
import nest_asyncio
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok, conf
from PIL import Image
import torch
import io
import os

nest_asyncio.apply()

from google.colab import userdata
conf.get_default().auth_token = userdata.get('NGROK_TOKEN')

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

DIST_PATH = "/content/Mills-Museum/src/frontend/mcam-keyword-generator/dist"

@app.get("/")
async def serve_frontend():
    return FileResponse(f"{DIST_PATH}/index.html")

app.mount("/assets", StaticFiles(directory=f"{DIST_PATH}/assets"), name="assets")

@app.post("/predict")
async def predict(file: UploadFile = File(...), term_count: int = Form(default=10)):
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    art_query = {"image": image, "text": ""}
    query_input = [art_query]

    image_features = embedding_model.process(query_input)
    image_features = torch.nn.functional.normalize(image_features, p=2, dim=1)
    query_embedding = image_features.cpu().float().squeeze().tolist()

    mmr_docs = vectorstore.max_marginal_relevance_search_by_vector(
        embedding=query_embedding,
        k=term_count,
        fetch_k=term_count * 4,
        lambda_mult=0.7,
    )

    labels = []
    input_docs = []
    for doc in mmr_docs:
        term_label = doc.metadata.get("term_label", doc.page_content.split("\n")[0])
        input_docs.append({"text": doc.page_content})
        labels.append(term_label)

    rerank_inputs = {
        "instruction": "Retrieve Art & Architecture Thesaurus terms relevant to the given image.",
        "query": art_query,
        "documents": input_docs,
        "fps": 1.0,
    }

    scores = reranking_model.process(rerank_inputs)
    sorted_results = sorted(zip(scores, labels), reverse=True)

    keywords = [
        {"label": label, "score": round(float(score) * 100, 1)}
        for score, label in sorted_results
    ]

    return {"keywords": keywords}

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")
print(f"Open this URL in your browser to use the app!")

# Start server
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [20376]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: NgrokTunnel: "https://stilliform-celine-plaintively.ngrok-free.dev" -> "http://localhost:8000"
Open this URL in your browser to use the app!
INFO:     144.91.51.54:0 - "OPTIONS /predict HTTP/1.1" 200 OK
INFO:     144.91.51.54:0 - "GET / HTTP/1.1" 200 OK
INFO:     144.91.51.54:0 - "GET /assets/index-CcnF5K1k.css HTTP/1.1" 200 OK
INFO:     144.91.51.54:0 - "GET /assets/index-DiynOvYC.js HTTP/1.1" 200 OK
INFO:     144.91.51.54:0 - "OPTIONS /predict HTTP/1.1" 200 OK
INFO:     144.91.51.54:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     144.91.51.54:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     144.91.51.54:0 - "OPTIONS /predict HTTP/1.1" 200 OK
INFO:     144.91.51.54:0 - "POST /predict HTTP/1.1" 200 OK
